In [1]:
# Turning a Large Language Model into a Chatbot

In [2]:
# Logging en HuggingFace
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
login(token=os.getenv("HF_MISTRAL"))

In [3]:
def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
    pad_token_id=tokenizer.eos_token_id,**generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [4]:
# Importando el modelo de HF (Mistral7B)
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id = "mistralai/Mistral-7B-v0.3"
mistral7b_tokenizer = AutoTokenizer.from_pretrained(model_id)
mistral7b = AutoModelForCausalLM.from_pretrained(
model_id, device_map="auto", dtype="auto")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [5]:
prompt = "Me gustaría visitar París, haz una lista de los sitios que debería visitar."
res = generate(mistral7b, mistral7b_tokenizer, prompt)
print(f"Resultado generate(): {res}")

KeyboardInterrupt: 

In [ ]:
# Como el modelo esta entrenado para predecir el siguiente token, es importante hacer ingenieria de "prompts" para que el modelo
# actue como un chatbot. En este caso podemos añadr una introduccion que ayude al bot a comportarse como un chatbot util respondiendo
# preguntas
lamine_introduction = """
Lamine es un chatbot increible. Sabe de todo y es muy util respondiendo preguntas.
"""
# Creamos el prompt concatenando la introducción anterior y utilizando etiquetas de roles; el "Yo: ... Lamine:" para indicar quien
# esta hablando
full_prompt = f"{lamine_introduction}Yo: {prompt}\nLamine: "

In [ ]:
extended_text = generate(mistral7b, mistral7b_tokenizer, full_prompt,
max_new_tokens=100)

answer = extended_text[len(full_prompt):].strip()
print(answer)

In [6]:
from LamineBot import LamineBot


In [7]:
lamine = LamineBot(mistral7b, mistral7b_tokenizer)
lamine.chat(prompt)
lamine.chat("Y que me puedes contar sobre el equipo de futbol de la ciudad?")

'Sí, el equipo de fútbol de París es el Paris Saint-Germain (PSG). Fue fundado en 1970 y juega en la Ligue 1, la primera división de Francia. El PSG es uno de los equipos más exitosos de Francia y ha ganado varios títulos nacionales y europeos.'

In [8]:
lamine.chat("Y eres consciente que Ferran Torres 'vendio' al FCB yendose al PSG y encima hizo declaraciones muy feas contra el FCB en una de sus primeras entrevistas? Que opinion te merece esto de Ferran Torres?")

'Sí, Ferran Torres es un jugador de fútbol español que juega en el Paris Saint-Germain. Recientemente, ha hecho declaraciones muy negativas sobre el FC Barcelona, su antiguo equipo. Esto es muy malo y no es el comportamiento que se espera de un jugador profesional.'